# ST-Reason · 10 Error analysis

Where does each wrong answer come from: the CV stage (the needed fact never made it into the memory), retrieval (it was in the memory but not retrieved), or reasoning (it was in front of the LLM and it still chose wrong)? **CPU only**, a few minutes. Run 08 first.

**No manual labels.** The plan's hand-labelled sample is replaced by two automatic methods:

1. **Oracle attribution (STAR, exact).** Each STAR question was answered from the CV memory and from the oracle memory built from ground truth, with the same LLM and strategy. If the CV answer is wrong but the oracle answer is right, the error is caused by perception; if both are wrong, perception is not the cause (reasoning, the question, or the strategy).
2. **Evidence tracing (both datasets, heuristic).** The content words of the correct option (minus words already in the question) are looked up in the memory and in the items retrieved for that question:
   - not in the memory → **perception** (the needed event or object was never extracted, or was described with other words),
   - in the memory but not among the retrieved items (strategy D) → **retrieval**,
   - in front of the LLM → **reasoning**,
   - unreadable reply → **format**; a correct option with no distinctive words → **uninformative** (left out of the shares).

Method 2 is checked against method 1 on STAR (section 4): if tracing works, "perception" errors should be fixed by the oracle much more often than "reasoning" errors.

| Output in `outputs/error_analysis/` | Content |
| --- | --- |
| `error_categories.csv`, `error_shares.csv` | one row per wrong answer with its category; shares per run and question type |
| `oracle_attribution.csv` | STAR: perception vs non-perception errors, exact |
| `vlm_overlap.csv` | which questions the memory pipeline and the frames-only VLM each get right |
| `error_examples.md`, `fig7_error_examples.pdf/png` | worked examples with frames, memory items, question and answers |

In [1]:
# --- Parameters ----------------------------------------------------------------------------------
EVAL_SHARDS = [0, 1]
N_SHARDS = 4
RUNS = ['main.M4.D', 'main.M4.A', 'main.M1.D', 'star.M4.D', 'star.M4.A', 'star.ORACLE_M4.D',
        'main.VLM.F16', 'star.VLM.F16', 'main.BLIND.A']

In [2]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '10_error_analysis'
ST_QUIET = True
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB
import pandas as pd, numpy as np, warnings, textwrap, io
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
MANIFEST_PATH = P['runs'] / 'manifest_10.csv'
DRIVE_LOG_PATH = LOG_PATH
LOG_PATH = P['local'] / 'logs' / LOG_PATH.name
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
MET = P['metrics'] if P['metrics'].exists() else P['pred']
ERR = P['errors'] if P['errors'].exists() else MET            # files only, no new folders on the shared Drive
FIG = P['figures'] if P['figures'].exists() else ERR
WORK = P['local'] / 'errors'
WORK.mkdir(parents=True, exist_ok=True)
RUN_LOG = Run(params={'step': 'error_analysis', 'runs': RUNS})
def shard_of(uid):
    return int(hashlib.sha1(str(uid).encode()).hexdigest(), 16) % N_SHARDS
def save_out(obj, name, folder=None):
    folder = folder or ERR
    tmp = WORK / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(tmp, index=False)
    else:
        tmp.write_text(obj)
    shutil.copyfile(tmp, folder / name)

Mounted at /content/drive
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 2a9e0e4f87
[07:18:41] INFO  run started: 10_error_analysis-20261019-071835-987517


## 1. Answers, questions and memories

Answers are read from the answer zips (they keep the retrieved and cited item IDs, which `answers_all.csv` does not). Memories come from the newest notebook-04 zip.

In [3]:
def zip_time(p):
    m = re.search(r'_(\d{8}-\d{6})_', p.name)
    return m.group(1) if m else ''
rows, seen = [], set()
for z in sorted(P['pred'].glob('0[567]_*.zip'), key=zip_time, reverse=True):
    try:
        with zipfile.ZipFile(z) as zf:
            for m in zf.namelist():
                if m.split('/')[0] not in RUNS or not m.endswith('.jsonl'):
                    continue
                for line in zf.read(m).decode().splitlines():
                    if line.strip():
                        r = json.loads(line)
                        if (r['run'], r['uid']) not in seen:
                            seen.add((r['run'], r['uid'])); rows.append(r)
    except (zipfile.BadZipFile, OSError):
        pass
A = pd.DataFrame(rows)
A = A[A.uid.map(lambda u: shard_of(str(u).split('__')[0])).isin(EVAL_SHARDS)].reset_index(drop=True)
for c in ['retrieved', 'retrieved_2', 'evidence', 'raw', 'valid_json', 'strategy', 'format']:
    if c not in A:
        A[c] = None
print(A.groupby('run').size().rename('answers').to_string())

def read_set(name):
    f = P['subsets'] / f'{name}.csv'
    return pd.read_csv(f, dtype={'video_id': str}) if f.exists() else pd.DataFrame()
Q = {}
for name in ['main_grid_eval', 'star_main']:
    for r in read_set(name).to_dict('records'):
        n = int(r.get('n_options', 5) or 5) if name == 'star_main' else 5
        opts = [str(r[f'a{i}']).rstrip('.') for i in range(n) if f'a{i}' in r and pd.notna(r[f'a{i}'])]
        Q[r['uid']] = {'question': str(r['question']), 'options': opts, 'gold': int(r['answer_idx']),
                       'type': r.get('group') if name == 'main_grid_eval' else r.get('type'), 'video': str(r['video_id']),
                       'dataset': 'nextqa' if name == 'main_grid_eval' else 'star'}

MEM = {}
mem_zips = sorted(P['memory'].glob('04_memories_*.zip'))
need = {(f, k) for f, k in zip(A.format, A.key) if isinstance(f, str) and re.match(r'^(ORACLE_)?M[1-4]$', f)}
need |= {('M4', k) for k in A.key if isinstance(k, str)} | {('ORACLE_M4', k) for k in A.key if isinstance(k, str) and k.startswith('star_')}
if mem_zips:
    with zipfile.ZipFile(mem_zips[-1]) as zf:
        names = set(zf.namelist())
        for f, k in need:
            if f'{f}/{k}.json' in names:
                MEM[(f, k)] = json.loads(zf.read(f'{f}/{k}.json'))['items']
print(f'questions: {len(Q)} | memories loaded: {len(MEM)} from {mem_zips[-1].name if mem_zips else "(none)"}')

run
main.BLIND.A        193
main.M1.D           193
main.M4.A           193
main.M4.D           193
main.VLM.F16        193
star.M4.A           190
star.M4.D           190
star.ORACLE_M4.D     94
star.VLM.F16        190
questions: 760 | memories loaded: 630 from 04_memories_20261017-124338.zip


## 2. Evidence tracing

In [4]:
STOP = {'a', 'an', 'the', 'some', 'something', 'somewhere', 'from', 'on', 'in', 'into', 'with', 'of', 'at', 'to', 'up',
        'person', 'someone', 'none', 'and', 'or', 'it', 'is', 'his', 'her', 'their', 'they', 'he', 'she', 'what', 'why',
        'how', 'did', 'does', 'do', 'was', 'were', 'be', 'after', 'before', 'while', 'when', 'video', 'man', 'woman',
        'lady', 'boy', 'girl', 'people', 'that', 'this', 'there', 'for', 'as', 'by', 'are', 'its', 'one', 'other'}
def words(s):
    out = set()
    for w in re.findall(r'[a-z]+', str(s).lower().replace('_', ' ')):
        for suf in ('ing', 'es', 's', 'ed'):
            if w.endswith(suf) and len(w) > len(suf) + 2:
                w = w[:-len(suf)]; break
        if len(w) > 1:
            out.add(w)
    return out - STOP
TIMED = ('event', 'state', 'relation', 'sentence', 'caption', 'object')
def item_words(items, ids=None):
    w = set()
    for it in items or []:
        if it.get('kind') in TIMED and (ids is None or it['id'] in ids):
            w |= words(it['text'])
    return w

def classify(r):
    q = Q.get(str(r.uid).split('__')[0])
    if q is None:
        return None
    if r.correct:
        return 'correct'
    if not r.valid_json:
        return 'format'
    gold = words(q['options'][q['gold']]) - words(q['question'])
    if not gold:
        return 'uninformative'
    fmt = r.format if isinstance(r.format, str) else ''
    items = MEM.get((fmt, r.key)) if fmt.startswith(('M', 'ORACLE')) else None
    if items is None:
        return 'no memory (baseline)'
    if not gold & item_words(items):
        return 'perception'
    if r.strategy in ('D', 'D-X') and isinstance(r.retrieved, list):
        shown = set(r.retrieved) | set(r.retrieved_2 if isinstance(r.retrieved_2, list) else [])
        if not gold & item_words(items, shown):
            return 'retrieval'
    return 'reasoning'

A['category'] = [classify(r) for r in A.itertuples()]
A['dataset'] = A.uid.map(lambda u: Q.get(str(u).split('__')[0], {}).get('dataset'))
A['qtype'] = A.uid.map(lambda u: Q.get(str(u).split('__')[0], {}).get('type'))
W = A[A.category.notna() & (A.category != 'correct')]
save_out(W[['run', 'uid', 'key', 'qtype', 'pred', 'gold', 'category']], 'error_categories.csv')
CATS = ['perception', 'retrieval', 'reasoning', 'format', 'uninformative', 'no memory (baseline)']
sh = (W.groupby('run').category.value_counts(normalize=True).unstack().reindex(columns=CATS).fillna(0))
sh.insert(0, 'errors', W.groupby('run').size())
sh.insert(0, 'accuracy', A.groupby('run').correct.mean())
save_out(sh.round(3).reset_index(), 'error_shares.csv')
print('Share of each cause among the wrong answers (rows sum to 1 over the category columns):\n')
print(sh.round(3).to_string())
bt = (W[W.run.isin(['main.M4.D', 'star.M4.D'])].groupby(['run', 'qtype']).category.value_counts(normalize=True)
      .unstack().reindex(columns=CATS[:5]).fillna(0).round(3))
print('\nby question type (M4 + D):\n', bt.to_string())
save_out(bt.reset_index(), 'error_shares_by_type.csv')

Share of each cause among the wrong answers (rows sum to 1 over the category columns):

category          accuracy  errors  perception  retrieval  reasoning  format  uninformative  no memory (baseline)
run                                                                                                              
main.BLIND.A         0.420     112       0.000      0.000      0.000   0.000            0.0                   1.0
main.M1.D            0.332     129       0.287      0.000      0.147   0.566            0.0                   0.0
main.M4.A            0.544      88       0.591      0.000      0.295   0.114            0.0                   0.0
main.M4.D            0.575      82       0.634      0.049      0.305   0.012            0.0                   0.0
main.VLM.F16         0.829      33       0.000      0.000      0.000   0.000            0.0                   1.0
star.M4.A            0.516      92       0.772      0.000      0.207   0.022            0.0                   0.0


## 3. Oracle attribution on STAR (exact)

Questions answered by both `star.M4.D` (CV memory) and `star.ORACLE_M4.D` (ground-truth memory), same LLM and strategy.

In [5]:
OA = None
if {'star.M4.D', 'star.ORACLE_M4.D'} <= set(A.run):
    cv = A[A.run == 'star.M4.D'].set_index('uid'); orc = A[A.run == 'star.ORACLE_M4.D'].set_index('uid')
    common = cv.index.intersection(orc.index)
    c, o = cv.loc[common, 'correct'].astype(bool), orc.loc[common, 'correct'].astype(bool)
    OA = pd.DataFrame({'outcome': ['both right', 'CV wrong, oracle right (perception error)',
                                   'both wrong (not perception)', 'CV right, oracle wrong'],
                       'questions': [int((c & o).sum()), int((~c & o).sum()), int((~c & ~o).sum()), int((c & ~o).sum())]})
    OA['share'] = (OA.questions / len(common)).round(3)
    n_err = int((~c).sum())
    print(OA.to_string(index=False))
    print(f'\nOf the {n_err} wrong CV-memory answers, {int((~c & o).sum())} ({(~c & o).sum() / max(1, n_err):.0%}) are fixed by '
          f'perfect perception: perception causes most errors; the other {(~c & ~o).sum() / max(1, n_err):.0%} would remain.')
    save_out(OA, 'oracle_attribution.csv')
else:
    print('star.M4.D or star.ORACLE_M4.D missing: skipped')

                                  outcome  questions  share
                               both right         43  0.457
CV wrong, oracle right (perception error)         43  0.457
              both wrong (not perception)          5  0.053
                   CV right, oracle wrong          3  0.032

Of the 48 wrong CV-memory answers, 43 (90%) are fixed by perfect perception: perception causes most errors; the other 10% would remain.


## 4. Does evidence tracing agree with the oracle?

On STAR, for the CV-memory errors that tracing labels *perception* vs *retrieval or reasoning*, how often does the oracle memory fix the question? If tracing works, the oracle fixes far more of the "perception" errors.

In [6]:
if OA is not None:
    t = A[A.run == 'star.M4.D'].set_index('uid')
    t = t[t.category.isin(['perception', 'retrieval', 'reasoning'])]
    fixed = orc.correct.reindex(t.index)
    t = t.assign(fixed_by_oracle=fixed).dropna(subset=['fixed_by_oracle'])
    chk = (t.assign(group=np.where(t.category == 'perception', 'traced: perception', 'traced: retrieval or reasoning'))
           .groupby('group').fixed_by_oracle.agg(['size', 'mean']).rename(columns={'size': 'errors', 'mean': 'fixed_by_oracle'}))
    print(chk.round(3).to_string())
    save_out(chk.round(3).reset_index(), 'tracing_vs_oracle.csv')
    print('\nRead: if "perception" errors are fixed by the oracle clearly more often, the word-tracing heuristic is a usable '
          'proxy on NExT-QA, where no oracle exists. If not, report only the oracle attribution.')

                                errors fixed_by_oracle
group                                                 
traced: perception                  39        0.871795
traced: retrieval or reasoning       9             1.0

Read: if "perception" errors are fixed by the oracle clearly more often, the word-tracing heuristic is a usable proxy on NExT-QA, where no oracle exists. If not, report only the oracle attribution.


## 5. Memory pipeline vs frames-only VLM

Which questions each system gets right. Questions the pipeline solves and the VLM does not are where structured memory adds something the frames alone did not give.

In [7]:
VO = []
for ds in ['main', 'star']:
    a, b = f'{ds}.M4.D', f'{ds}.VLM.F16'
    if {a, b} <= set(A.run):
        x = A[A.run == a].set_index('uid').correct.astype(bool); y = A[A.run == b].set_index('uid').correct.astype(bool)
        com = x.index.intersection(y.index); x, y = x[com], y[com]
        VO.append({'dataset': ds, 'n': len(com), 'both right': int((x & y).sum()), 'memory only': int((x & ~y).sum()),
                   'VLM only': int((~x & y).sum()), 'both wrong': int((~x & ~y).sum()),
                   'oracle-style upper bound (either right)': round(float((x | y).mean()), 3)})
VO = pd.DataFrame(VO)
if len(VO):
    print(VO.to_string(index=False))
    save_out(VO, 'vlm_overlap.csv')

dataset   n  both right  memory only  VLM only  both wrong  oracle-style upper bound (either right)
   main 193         106            5        54          28                                    0.855
   star 190          92           10        52          36                                    0.811


## 6. Worked examples

Picked automatically (first matching question, fixed order, so the choice is reproducible): a STAR perception error fixed by the oracle, a NExT-QA retrieval error, a NExT-QA reasoning error, and a question the memory pipeline gets right but the frames-only VLM gets wrong. Frames are the key frames notebook 03 saved for windows with events.

In [8]:
CVP = sorted(P['cv'].glob('03*_shard*.zip'), key=zip_time, reverse=True)
_FIDX = {}
for z in CVP:
    try:
        with zipfile.ZipFile(z) as zf:
            for m in zf.namelist():
                if m.startswith('frames/'):
                    _FIDX.setdefault(m, z)
    except (zipfile.BadZipFile, OSError):
        pass
def frames_of(key, k=3):
    ms = sorted(m for m in _FIDX if m.startswith(f'frames/{key}/'))
    if not ms:
        return []
    pick = [ms[int(i)] for i in np.linspace(0, len(ms) - 1, min(k, len(ms))).round()]
    out = []
    for m in dict.fromkeys(pick):
        with zipfile.ZipFile(_FIDX[m]) as zf:
            out.append(Image.open(io.BytesIO(zf.read(m))).convert('RGB'))
    return out

def items_text(fmt, key, ids=None, limit=6):
    its = [it for it in MEM.get((fmt, key), []) if it.get('kind') in TIMED and (ids is None or it['id'] in ids)]
    return [it['text'] for it in its[:limit]]

def pick(run, cond):
    d = A[A.run == run]
    for r in d.sort_values('uid').itertuples():
        if cond(r):
            return r
    return None
correct_of = lambda run: A[A.run == run].set_index('uid').correct.astype(bool).to_dict()
orc_ok, vlm_main = correct_of('star.ORACLE_M4.D'), correct_of('main.VLM.F16')
vlm_preds = {**A[A.run == 'main.VLM.F16'].set_index('uid').pred.to_dict(), **A[A.run == 'star.VLM.F16'].set_index('uid').pred.to_dict()}
EX = [('STAR: perception error, fixed by perfect perception', 'star.M4.D',
       lambda r: r.category == 'perception' and orc_ok.get(r.uid) is True),
      ('NExT-QA: retrieval error', 'main.M4.D', lambda r: r.category == 'retrieval'),
      ('NExT-QA: reasoning error (evidence was retrieved)', 'main.M4.D', lambda r: r.category == 'reasoning'),
      ('NExT-QA: memory pipeline right, frames-only VLM wrong', 'main.M4.D',
       lambda r: bool(r.correct) and vlm_main.get(r.uid) is False)]
md_lines, panels = ['# Error analysis: worked examples', ''], []
for title, run, cond in EX:
    r = pick(run, cond)
    if r is None:
        md_lines += [f'## {title}', '', '(no matching question)', '']; continue
    q = Q[str(r.uid).split('__')[0]]
    opts = '; '.join(f'{"ABCDE"[i]}. {o}' for i, o in enumerate(q['options']))
    shown = r.retrieved if isinstance(r.retrieved, list) else None
    mem = items_text('M4', r.key, set(shown) if shown else None)
    orc = items_text('ORACLE_M4', r.key) if r.key.startswith('star_') else []
    md_lines += [f'## {title}', '', f'- **Question** ({r.uid}, {q["type"]}): {q["question"]}', f'- **Options:** {opts}',
                 f'- **Correct:** {r.gold} · **pipeline ({run}):** {r.pred} · **frames-only VLM:** {vlm_preds.get(r.uid, "—")}'
                 + (f' · **oracle memory:** {"right" if orc_ok.get(r.uid) else "wrong"}' if r.key.startswith('star_') else ''),
                 f'- **Category:** {r.category}', '- **Memory items shown to the LLM** (first 6):'] + \
                [f'  - `{t}`' for t in mem] + (['- **Oracle memory** (first 6):'] + [f'  - `{t}`' for t in orc] if orc else []) + ['']
    panels.append((title, r, q, mem, orc))
save_out('\n'.join(md_lines) + '\n', 'error_examples.md')
print('\n'.join(md_lines))

if panels:
    fig, axes = plt.subplots(len(panels), 4, figsize=(7.2, 1.55 * len(panels)), squeeze=False,
                             gridspec_kw={'width_ratios': [1, 1, 1, 2.7], 'wspace': 0.04, 'hspace': 0.25})
    for i, (title, r, q, mem, orc) in enumerate(panels):
        ims = frames_of(r.key)
        for j in range(3):
            ax = axes[i][j]; ax.axis('off')
            if j < len(ims):
                ax.imshow(ims[j])
        tx = axes[i][3]; tx.axis('off')
        gi = 'ABCDE'.index(r.gold) if isinstance(r.gold, str) and r.gold in 'ABCDE' else q['gold']
        pr = q['options']['ABCDE'.index(r.pred)] if isinstance(r.pred, str) and r.pred in 'ABCDE'[:len(q['options'])] else 'no valid answer'
        lines = textwrap.wrap('Q: ' + q['question'], 58) + textwrap.wrap('Correct: ' + q['options'][gi], 58) + \
                textwrap.wrap('Pipeline: ' + pr, 58) + textwrap.wrap('Memory: ' + (' | '.join(m[:55] for m in mem[:2]) or '(nothing relevant)'), 58)
        tx.text(0.02, 0.98, title, va='top', fontsize=7, weight='bold', color='#0b0b0b', transform=tx.transAxes)
        tx.text(0.02, 0.80, '\n'.join(lines[:7]), va='top', fontsize=6.4, color='#0b0b0b', transform=tx.transAxes, linespacing=1.3)
    for ext in ['pdf', 'png']:
        fig.savefig(WORK / f'fig7_error_examples.{ext}', dpi=200, bbox_inches='tight')
        shutil.copyfile(WORK / f'fig7_error_examples.{ext}', FIG / f'fig7_error_examples.{ext}')
    plt.close(fig)
    print('figure saved: fig7_error_examples')
RUN_LOG.finish('done', notes=f'{len(W)} wrong answers classified')

# Error analysis: worked examples

## STAR: perception error, fixed by perfect perception

- **Question** (star_Interaction_T1_2857, Interaction): Which object was put down by the person?
- **Options:** A. The clothes; B. The cup/glass/bottle; C. The blanket; D. The book
- **Correct:** C · **pipeline (star.M4.D):** D · **frames-only VLM:** C · **oracle memory:** right
- **Category:** perception
- **Memory items shown to the LLM** (first 6):
  - `E1 [t=0.0-3.0s] person_3 picks up backpack_2 (floor)`
  - `E2 [t=3.0-5.1s] person_3 holds backpack_2 (near curtain)`
  - `E3 [t=3.0-5.1s] person_3 adjusts backpack_2 (hands)`
  - `E4 [t=3.0-5.1s] person_3 moves backpack_2 (slightly)`
  - `E1 OVERLAPS E2`
  - `E2 OVERLAPS E3`
- **Oracle memory** (first 6):
  - `E1 [t=0.0-4.3s] person tidying up a blanket/s`
  - `E2 [t=0.0-4.3s] person holding a blanket`
  - `E3 [t=0.0-5.1s] person putting a towel/s somewhere`
  - `E4 [t=0.0-5.1s] person holding a bag`
  - `E5 [t=0.0-5.1s] person holding a towel/

## What to send back

The tables from sections 2–5. Also open `fig7_error_examples.png` in `outputs/figures/`: if an example is unclear, say which, and a different one can be picked.